# <center style="font-family: consolas; font-size: 32px; font-weight: bold;">  Evaluating LLM Applications Using LangChain </center>
<center style="font-family: consolas; font-size: 25px; font-weight: bold;">  (OpenRouter + Google Colab + LangChain 1.x + Your Own Data) </center>

***

When constructing a sophisticated application employing an LLM, a crucial yet challenging aspect revolves around evaluating its performance. How can you ascertain if it meets accuracy standards?

Moreover, if you opt to alter your implementation — perhaps by substituting a different LLM or adjusting the strategy for utilizing a vector database or other retrieval mechanisms — how can you gauge whether these changes enhance or detract from the application?

This notebook discusses the challenges of evaluating the performance of applications built with large language models (LLMs) and explores strategies for effectively assessing their accuracy and effectiveness. It emphasizes the importance of understanding the inputs and outputs of each step in the application's workflow, and explores using language models themselves to evaluate other models and applications.

**This version has been fully modernized and generalized:**

| Old (deprecated / hardcoded) | New |
|---|---|
| Hardcoded Kaggle path to one specific PDF | **Upload your own PDF directly in Colab** — works with any document |
| Direct OpenAI + `kaggle_secrets` | **OpenRouter**, so you can use any chat/embedding model with one key |
| `VectorstoreIndexCreator` + `RetrievalQA` (deprecated) | `InMemoryVectorStore` + `create_stuff_documents_chain` + `create_retrieval_chain` (current LCEL-based RAG pattern) |
| `langchain.chat_models.ChatOpenAI` / `langchain.document_loaders` / `langchain.vectorstores` / `langchain.embeddings` (deprecated import paths) | `langchain_openai`, `langchain_community.document_loaders`, `langchain_core.vectorstores` |
| `QAGenerateChain` (deprecated) | A small **Pydantic + `with_structured_output()`** helper that generates question/answer pairs from your documents |
| `QAEvalChain` (deprecated) | A small **Pydantic + `with_structured_output()`** helper that grades predictions as CORRECT/INCORRECT |
| `langchain.debug = True` (old global flag) | `langchain.globals.set_debug(True)` (current API) |

#### <a id="top"></a>
# <div style="box-shadow: rgb(60, 121, 245) 0px 0px 0px 3px inset, rgb(255, 255, 255) 10px -10px 0px -3px, rgb(31, 193, 27) 10px -10px, rgb(255, 255, 255) 20px -20px 0px -3px, rgb(255, 217, 19) 20px -20px, rgb(255, 255, 255) 30px -30px 0px -3px, rgb(255, 156, 85) 30px -30px, rgb(255, 255, 255) 40px -40px 0px -3px, rgb(255, 85, 85) 40px -40px; padding:20px; margin-right: 40px; font-size:30px; font-family: consolas; text-align:center; display:fill; border-radius:15px; color:rgb(60, 121, 245);"><b>Table of contents</b></div>

<div style="background-color: rgba(60, 121, 245, 0.03); padding:30px; font-size:15px; font-family: consolas;">
<ul>
    <li><a href="#1" target="_self" rel=" noreferrer nofollow">1. Setting Up Working Environment & Loading Your Data </a></li>
    <li><a href="#2" target="_self" rel=" noreferrer nofollow">2. Manual Evaluation & Debugging </a></li>
    <li><a href="#3" target="_self" rel=" noreferrer nofollow">3. LLM-Assisted Evaluation </a></li>
    <li><a href="#4" target="_self" rel=" noreferrer nofollow">4. Observing Behind the Scenes </a></li>
    <li><a href="#5" target="_self" rel=" noreferrer nofollow">5. Grading Predictions with an LLM </a></li>
</ul>
</div>

***


<a id="1"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 1. Setting Up Working Environment & Loading Your Data </b></div>

As usual, we'll start by installing the packages we need and setting up OpenRouter for both **chat** and **embeddings** (OpenRouter now supports an OpenAI-compatible `/embeddings` endpoint too, so we can use one key/client for everything).


In [1]:
# Install required libraries (Colab)
!pip install -q openai langchain langchain-core langchain-community langchain-openai langchain-classic langchain-text-splitters pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### OpenRouter API Key

Create an account at [openrouter.ai](https://openrouter.ai/keys) and get an API key.

On Colab, store the key in **Secrets** (the 🔑 icon on the left sidebar) under the name `OPENROUTER_API_KEY`. If the secret isn't found, you'll be prompted to enter it manually.


In [2]:
import os

try:
    # If you're running this on Google Colab
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = None

if not OPENROUTER_API_KEY:
    import getpass
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY


In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Pick any chat model available on OpenRouter: https://openrouter.ai/models
llm_model = "openai/gpt-4o-mini"

# Pick any embedding model available on OpenRouter: https://openrouter.ai/models?fmt=cards&output_modalities=embeddings
embedding_model_name = "openai/text-embedding-3-small"

llm = ChatOpenAI(
    temperature=0.0,
    model=llm_model,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)

embedding_model = OpenAIEmbeddings(
    model=embedding_model_name,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)


### Upload YOUR data

The original notebook used one hardcoded PDF from a Kaggle dataset path. This version lets you **upload any PDF of your own** directly into Colab — your data, not a fixed example file.

Run the cell below, then use the "Choose Files" button that appears to upload a PDF from your computer.


In [4]:
try:
    from google.colab import files
    print("Please upload a PDF file to use as your knowledge base...")
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"\nLoaded file: {pdf_path}")
except Exception as e:
    # Not running on Colab, or upload was skipped
    pdf_path = None
    print("Could not open the Colab upload widget. "
          "Set `pdf_path` manually to a local file path instead.")
    print(e)


Please upload a PDF file to use as your knowledge base...


Saving abu_khaled_rag.pdf to abu_khaled_rag.pdf

Loaded file: abu_khaled_rag.pdf


We need a chain to evaluate, so we'll build a document question-answering chain over your PDF. First, load and split it into chunks.


In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splits = text_splitter.split_documents(data)

print(f"Loaded {len(data)} page(s), split into {len(splits)} chunk(s).")


/tmp/ipykernel_1606/2357189296.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 2 page(s), split into 8 chunk(s).


Now we'll build the vector store and the retrieval-QA chain. The old `VectorstoreIndexCreator` + `RetrievalQA` combo is deprecated; the current, LCEL-based pattern is:

1. Embed and store the chunks in a vector store (`InMemoryVectorStore` — no extra database needed, great for prototyping and small documents).
2. Build a "combine documents" chain that stuffs retrieved chunks into a prompt (`create_stuff_documents_chain`).
3. Wrap it with a retriever into a full retrieval chain (`create_retrieval_chain`).


In [6]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

vectorstore = InMemoryVectorStore.from_documents(splits, embedding_model)
retriever = vectorstore.as_retriever()

system_prompt = (
    "You are an assistant answering questions about a document. "
    "Use only the following retrieved context to answer the question. "
    "If you don't know the answer based on the context, say you don't know. "
    "Keep the answer concise (max 3 sentences).\n\n"
    "Context:\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

combine_docs_chain = create_stuff_documents_chain(llm, qa_prompt)
qa_chain = create_retrieval_chain(retriever, combine_docs_chain)


Now that we have our application set up, we need to figure out what data points we want to evaluate it on.


<a id="2"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 2. Manual Evaluation & Debugging </b></div>

We'll start with the simplest method of debugging: coming up with data points we think are good examples and testing them manually. Look at a couple of chunks from **your own document** to get a sense of what's in there.


In [7]:
splits[0]


Document(metadata={'producer': 'Qt 5.15.13', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-07-27T22:41:00+00:00', 'title': '', 'source': 'abu_khaled_rag.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='ﺪﻟﺎﺧ\tﻮﺑأ\tﻢﻌﻄﻣﺪﻟﺎﺧ\tﻮﺑأ\tﻢﻌﻄﻣ \n2011\n\tمﺎﻋ\tﺬﻨﻣ\tﺔﻠﻴﺻﻷا\tمﻮﻴﻔﻟا\tﺔﻬﻜﻧ\ninstagram.com/abukhaled.fayoum\n\t\n\x01\n\t\t|\t\t\n01030685988\n\t\n\x01\n|\t\t\n\x01\nﻢﻌﻄﻤﻟا\tﻦﻋ\tﺔﻣﺎﻋ\tةﺮﻈﻧ\nﻢﻌﻄﻤﻟا\tﻦﻋ\tﺔﻣﺎﻋ\tةﺮﻈﻧ\t(\n1\n1\n.ﺺﺨﺷ\n40\n\tﱴﺣ\tﻊﺴﺘﺗ\tتﺎﺒﺳﺎﻨﻤﻠﻟ\tﺔﺻﺎﺧ\tﺔﻋﺎﻗ\tﱃإ\tﺔﻓﺎﺿﻹﺎﺑ\t،(ﺪﻌﻘﻣ\n50\n)\tﺔﻘﻳﺪﺤﻟا\tﲆﻋ\tﻞﻄﻣ\tﻲﺟرﺎﺧ\tساﺮﺗو\t(ﺪﻌﻘﻣ\n70\n)\tﺔﻔﻴﻜﻣ\tﺔﻴﻠﺧاد\tﺔﻟﺎﺻ\tﲆﻋ\tﺔﻋزﻮﻣ\tﳼﺮﻛ\n120\n\tدﺪﻌﻟ\tﻢﻌﻄﻤﻟا\tﻊﺴﺘﻳ\t.نورﺎﻗ\tةﲑﺤﺑ\tﻦﻣ\tﺔﺟزﺎﻄﻟا\tكﺎﻤﺳﻷاو\t،ﱵﻴﺒﻟا\tﻦﺟاﻮﻄﻟاو\t،ﻢﺤﻔﻟا\tﲆﻋ\tتﺎﻳﻮﺸﻤﻟا\tقﺎﺒﻃﺄﺑ\tﺮﻬﺘﺸﻳو\t،ﻦﻤﺣﺮﻟا\tﺪﺒﻋ\tﺪﻟﺎﺧ\tﻒﻴﺸﻟا\tﺪﻳ\tﲆﻋ\n2011\n\tمﺎﻋ\tﺲﺳﺄﺗ\t،مﻮﻴﻔﻟا\tﺔﻨﻳﺪﻣ\tﺐﻠﻗ\tﻲﻓ\tﻞﻴﺻﻷا\tﻲﻗﴩﻟاو\tيﴫﻤﻟا\tﺦﺒﻄﻤﻟا\tمّﺪﻘﻳ\tﲇﺋﺎﻋ\tﻢﻌﻄﻣ\tﻮﻫ\tﺪﻟﺎﺧ\tﻮﺑأ\tﻢﻌﻄﻣ\n.مﻮﻴﻔﻟا\tﺔﻨﻳﺪﻣ\tقﺎﻄﻧ\tﻞﺧاد\tﴍﺎﺒﻣ\tﻞﻴﺻﻮﺗ\tﻚﻟﺬﻛو\t(بﻮﺷﺎﺘﺴﻧإ\t،تﺎﺒﻠﻃ)\tتﺎﻘﻴﺒﻄﺗ\tﱪﻋ\tﻞﻴﺻﻮﺗ\tﺔﻣﺪﺧ\tمّﺪﻘﻳو\t،ﻢﻴﻴﻘﺗ\t\n2100\n\tﻦﻣ\tﱶﻛ

In [8]:
splits[min(3, len(splits) - 1)]


Document(metadata={'producer': 'Qt 5.15.13', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-07-27T22:41:00+00:00', 'title': '', 'source': 'abu_khaled_rag.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='(ﻪﻴﻨﺟ)\tﺮﻌﺴﻟا\nيﻮﺸﻣ\tﻲﻄﻠﺑ\nنزﻮﻟا\tﺐﺴﺣ\nﻮﻠﻴﻛ\t/\t\n140\nﲇﻘﻣ\tيرﻮﺑ\nنزﻮﻟا\tﺐﺴﺣ\nﻮﻠﻴﻛ\t/\t\n180\nﻂﺳﻮﺘﻣ\tرﺎﺣ\n\tيﻮﺸﻣ\tيﱪﻤﺟ\nﻞﺒﺘﻣ\tيﱪﻤﺟ\tماﺮﺟ\t\n250\n280\nنﺮﻔﻟﺎﺑ\tﻚﻤﺳ\tﺔﻴﻨﻴﺻ\nزرﻷاو\tﻢﻃﺎﻤﻄﻟا\tﺔﺼﻠﺼﺑ\tﻞﻜﺸﻣ\tﻚﻤﺳ\n260\nتﺎﻳﻮﻠﺤﻟا\nتﺎﻳﻮﻠﺤﻟا\nﻒﻨﺼﻟا\nﻒﻨﺼﻟا\n(ﻪﻴﻨﺟ)\tﺮﻌﺴﻟا\n(ﻪﻴﻨﺟ)\tﺮﻌﺴﻟا\nﲇﻋ\tمأ\n65\nﺔﻄﺸﻘﻟﺎﺑ\tﺔﻓﺎﻨﻛ\n70\nﱭﻠﺑ\tزر\n40\nﺔﺳﻮﺒﺴﺑ\n35\nتﺎﺑوﴩﻤﻟا\nتﺎﺑوﴩﻤﻟا\nﻒﻨﺼﻟا\nﻒﻨﺼﻟا\n(ﻪﻴﻨﺟ)\tﺮﻌﺴﻟا\n(ﻪﻴﻨﺟ)\tﺮﻌﺴﻟا\n(عﺎﻨﻌﻨﻟﺎﺑ\tنﻮﻤﻴﻟ\t/\tﺔﻟواﺮﻓ\t/\tﻮﺠﻧﺎﻣ)\tﺔﺟزﺎﻃ\tﺮﺋﺎﺼﻋ\n35\nﺔﻳزﺎﻏ\tتﺎﺑوﴩﻣ\n20\nنﻮﺴﻨﻳ\t/\tﺔﻓﺮﻗ\t/\tيﺎﺷ\n15')

**Write 2-3 example questions and their correct ("ground truth") answers based on the actual content of the document you uploaded.** Replace the placeholder examples below with ones grounded in your own data — that's the whole point of manual evaluation: you, the human, are the source of truth here.


In [9]:
examples = [
    {
        "query": "REPLACE ME: a question whose answer is clearly in your document",
        "answer": "REPLACE ME: the correct/expected answer"
    },
    {
        "query": "REPLACE ME: a second question",
        "answer": "REPLACE ME: its correct answer"
    },
]


But this doesn't scale — it takes time to look through each example and figure out what's going on. A better way is to automate it.


<a id="3"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 3. LLM-Assisted Evaluation </b></div>

One way to automate the evaluation process is with LLMs themselves: use a language model to generate question/answer pairs directly from your document's chunks.

The original notebook used LangChain's `QAGenerateChain` for this — that class is now deprecated. The modern equivalent is a **Pydantic schema + `with_structured_output()`**, the same pattern we used in the output-parsing notebook.


In [10]:
from pydantic import BaseModel, Field


class QAPair(BaseModel):
    query: str = Field(description="A specific question that can be answered using only the given document excerpt.")
    answer: str = Field(description="The correct answer to the question, based only on the document excerpt.")


qa_generator = llm.with_structured_output(QAPair)


Now let's generate a question/answer pair for the first few chunks of your document — this saves us from having to write every example by hand.


In [11]:
new_examples = []

for doc in splits[:5]:
    generation_prompt = (
        "Generate one specific, factual question-and-answer pair "
        "based only on the following document excerpt:\n\n"
        f"{doc.page_content}"
    )
    qa_pair = qa_generator.invoke(generation_prompt)
    new_examples.append({"query": qa_pair.query, "answer": qa_pair.answer})

new_examples[0]


{'query': 'What is the Instagram handle mentioned in the document?',
 'answer': 'instagram.com/abukhaled.fayoum'}

We just generated a bunch of question-answer pairs automatically — we didn't have to write them all ourselves, which saves time and lets us test more cases. Let's add these to the examples we created manually.


In [12]:
examples += new_examples
examples


[{'query': 'REPLACE ME: a question whose answer is clearly in your document',
  'answer': 'REPLACE ME: the correct/expected answer'},
 {'query': 'REPLACE ME: a second question',
  'answer': 'REPLACE ME: its correct answer'},
 {'query': 'What is the Instagram handle mentioned in the document?',
  'answer': 'instagram.com/abukhaled.fayoum'},
 {'query': 'What is the email address provided in the document?',
  'answer': 'info@abukhaled-fayoum.com'},
 {'query': "What is the value associated with the item labeled 'قووﺎﻃ  +  ﺔﺘﻔﻛ  +  بﺎﺒﻛ'?",
  'answer': '260'},
 {'query': "What is the value associated with the term 'ﻪﻴﻨﺟ' in the document?",
  'answer': '140'},
 {'query': 'What is the phone number mentioned in the document?',
  'answer': '01030685988'}]

So we've got these examples now — but how do we actually evaluate what's going on? The first thing to do is just run one example through the chain and look at the output it produces.


In [13]:
qa_chain.invoke({"input": examples[0]["query"]})["answer"]


"I don't know."

When we input a query, we receive an answer. However, this approach limits our visibility into the chain's inner workings. What exact prompt is fed into the language model? Which documents does it fetch? In more complex chains with multiple steps, what intermediate results are generated? Simply observing the final answer often isn't sufficient for understanding potential issues within the chain. To address this, LangChain provides a debug utility.


<a id="4"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 4. Observing Behind the Scenes </b></div>


If we want to observe what's happening behind the scenes, we can turn on debug mode. The old `langchain.debug = True` global flag is deprecated — the current API is `langchain.globals.set_debug(True)`.


In [15]:
from langchain_core.globals import set_debug

set_debug(True)

qa_chain.invoke({"input": examples[0]["query"]})

[chain/start] [chain:retrieval_chain] Entering Chain run with input:
{
  "input": "REPLACE ME: a question whose answer is clearly in your document"
}
[chain/start] [chain:retrieval_chain > chain:RunnableAssign<context>] Entering Chain run with input:
{
  "input": "REPLACE ME: a question whose answer is clearly in your document"
}
[chain/start] [chain:retrieval_chain > chain:RunnableAssign<context> > chain:RunnableParallel<context>] Entering Chain run with input:
{
  "input": "REPLACE ME: a question whose answer is clearly in your document"
}
[chain/start] [chain:retrieval_chain > chain:RunnableAssign<context> > chain:RunnableParallel<context> > chain:retrieve_documents] Entering Chain run with input:
{
  "input": "REPLACE ME: a question whose answer is clearly in your document"
}
[chain/start] [chain:retrieval_chain > chain:RunnableAssign<context> > chain:RunnableParallel<context> > chain:retrieve_documents > chain:RunnableLambda] Entering Chain run with input:
{
  "input": "REPLACE ME

{'input': 'REPLACE ME: a question whose answer is clearly in your document',
 'context': [Document(id='40675f56-9486-4986-85d7-7db55810eed2', metadata={'producer': 'Qt 5.15.13', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-07-27T22:41:00+00:00', 'title': '', 'source': 'abu_khaled_rag.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='.ﺐﻠﻄﻟا\tﺪﻨﻋ\tﻢﺤﻟ\tنوﺪﺑ\tرﺎﻀﺨﻟا\tﻦﺟﺎﻃو\t،ﺔﻟﻮﺒﺘﻟا\t،شﻮﺘﻔﻟا\t،جﻮﻨﻏ\tﺎﺑﺎﺒﻟا\t،ﺺﻤﺤﻟا\tﻞﺜﻣ\tﺔﻴﺗﺎﺒﻧ\tقﺎﺒﻃأ\tﺮﻓﻮﺘﺗ\t،ﻢﻌﻧ\t:ج\n؟تارﺎﻴﺳ\tﻒﻗﻮﻣ\tﺪﺟﻮﻳ\tﻞﻫ\t:س\n؟تارﺎﻴﺳ\tﻒﻗﻮﻣ\tﺪﺟﻮﻳ\tﻞﻫ\t:س\n.ةﴍﺎﺒﻣ\tﻢﻌﻄﻤﻟا\tمﺎﻣأ\tةرﺎﻴﺳ\t\n25\n\tﱄاﻮﺤﻟ\tﻊﺴﺘﻳ\tﻲﻧﺎﺠﻣ\tتارﺎﻴﺳ\tﻒﻗﻮﻣ\tﺮﻓﻮﺘﻳ\t:ج\n؟ﺔﺻﺎﺧ\tتﺎﺒﺳﺎﻨﻣ\tوأ\tدﻼﻴﻣ\tدﺎﻴﻋأ\tتﻼﻔﺣ\tﻢﻴﻈﻨﺗ\tﻦﻜﻤﻳ\tﻞﻫ\t:س\n؟ﺔﺻﺎﺧ\tتﺎﺒﺳﺎﻨﻣ\tوأ\tدﻼﻴﻣ\tدﺎﻴﻋأ\tتﻼﻔﺣ\tﻢﻴﻈﻨﺗ\tﻦﻜﻤﻳ\tﻞﻫ\t:س\n.ﻞﻴﺻﺎﻔﺘﻠﻟ\t\n01030685988\n\tﲆﻋ\tﻞﺻاﻮﺘﻟا\tﻦﻜﻤﻳو\t،(ﺔﺑﻮﻠﻄﻤﻟا\tﺔﻤﺋﺎﻘﻟاو\tفﻮﻴﻀﻟا\tدﺪﻋ\tﺐﺴﺣ\tﻒﻠﺘﺨﺗ)\tﻪﻴﻨﺟ\n1500\n\tﻦﻣ\tأﺪﺒﺗ\tتﺎﻗﺎﺑ\tﻞﻤﺸﺗو\tتﺎﺒﺳﺎﻨﻤﻠﻟ\tﺰﺠﺤﻠﻟ\tﺔﺣﺎﺘﻣ\tﺔﺻﺎﺨﻟا\tﺔﻋﺎﻘﻟا\t،ﻢﻌﻧ\t:ج\n؟لﻼﺣ\tتﺎﺒﺟو\tمﺪﻘﻳ\tﻢﻌﻄﻤﻟا\tﻞﻫ\t:س\n؟لﻼﺣ\tتﺎﺒﺟو\tمﺪﻘﻳ\tﻢﻌﻄﻤﻟا\tﻞﻫ\t:س\n.ﻦﻳ

When examining the output closely, you'll see it first goes into the retrieval step, then the "combine documents" step, then the LLM chain itself, where you can see the full prompt: the system message with the retrieved context stuffed in, plus your original question. In question-answering scenarios, errors often arise not from the language model itself but from flaws in the retrieval step — so scrutinizing exactly what context was retrieved is a key debugging tool.

Let's turn debug mode back off before running the full evaluation loop (otherwise the output gets very noisy).


In [16]:
set_debug(False)


Now let's create predictions for all our examples by looping through the chain.


In [17]:
predictions = []

for example in examples:
    result = qa_chain.invoke({"input": example["query"]})
    predictions.append({
        "query": example["query"],
        "answer": example["answer"],
        "result": result["answer"],
    })

predictions


[{'query': 'REPLACE ME: a question whose answer is clearly in your document',
  'answer': 'REPLACE ME: the correct/expected answer',
  'result': "I don't know."},
 {'query': 'REPLACE ME: a second question',
  'answer': 'REPLACE ME: its correct answer',
  'result': "I don't know."},
 {'query': 'What is the Instagram handle mentioned in the document?',
  'answer': 'instagram.com/abukhaled.fayoum',
  'result': 'The Instagram handle mentioned in the document is instagram.com/abukhaled.fayoum.'},
 {'query': 'What is the email address provided in the document?',
  'answer': 'info@abukhaled-fayoum.com',
  'result': 'The email address provided in the document is info@abukhaled-fayoum.com.'},
 {'query': "What is the value associated with the item labeled 'قووﺎﻃ  +  ﺔﺘﻔﻛ  +  بﺎﺒﻛ'?",
  'answer': '260',
  'result': "The value associated with the item labeled 'قووﺎﻃ  +  ﺔﺘﻔﻚ  +  بﺎﺒﻛ' is 260."},
 {'query': "What is the value associated with the term 'ﻪﻴﻨﺟ' in the document?",
  'answer': '140',
  '

<a id="5"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 5. Grading Predictions with an LLM </b></div>


With predictions in hand, let's evaluate them automatically. The original notebook used LangChain's `QAEvalChain` — also deprecated. We'll replace it with the same Pydantic + `with_structured_output()` pattern: define a `Grade` schema, bind it to the LLM, and ask it to compare each prediction against the ground-truth answer.


In [18]:
from typing import Literal


class Grade(BaseModel):
    grade: Literal["CORRECT", "INCORRECT"] = Field(
        description="Whether the predicted answer is factually consistent with the real answer."
    )
    reasoning: str = Field(
        description="A brief (1-2 sentence) explanation for the grade."
    )


grading_llm = llm.with_structured_output(Grade)

grading_prompt = """You are grading the accuracy of a predicted answer to a question, \
compared to a real (ground-truth) answer.

QUESTION: {query}

REAL ANSWER: {answer}

PREDICTED ANSWER: {result}

Grade the predicted answer as CORRECT if it is factually consistent with the real answer \
(it does not need to match word-for-word), or INCORRECT if it contradicts or misses the \
key facts in the real answer."""


In [19]:
graded_outputs = []

for p in predictions:
    grade = grading_llm.invoke(grading_prompt.format(**p))
    graded_outputs.append(grade)


Now let's loop through each example and print out the question, the real answer, the predicted answer, and the grade — this gives us a clear picture of where the chain is doing well and where it's struggling.


In [20]:
for i, (p, g) in enumerate(zip(predictions, graded_outputs)):
    print(f"Example {i}:")
    print("Question: " + p["query"])
    print("Real Answer: " + p["answer"])
    print("Predicted Answer: " + p["result"])
    print("Predicted Grade: " + g.grade)
    print("Reasoning: " + g.reasoning)
    print()


Example 0:
Question: REPLACE ME: a question whose answer is clearly in your document
Real Answer: REPLACE ME: the correct/expected answer
Predicted Answer: I don't know.
Predicted Grade: INCORRECT
Reasoning: The predicted answer 'I don't know' does not provide any information or attempt to address the question, which is inconsistent with the expectation of a correct answer.

Example 1:
Question: REPLACE ME: a second question
Real Answer: REPLACE ME: its correct answer
Predicted Answer: I don't know.
Predicted Grade: CORRECT
Reasoning: The predicted answer 'I don't know' does not contradict the real answer and is a valid response when the information is unknown.

Example 2:
Question: What is the Instagram handle mentioned in the document?
Real Answer: instagram.com/abukhaled.fayoum
Predicted Answer: The Instagram handle mentioned in the document is instagram.com/abukhaled.fayoum.
Predicted Grade: CORRECT
Reasoning: The predicted answer accurately reflects the real answer by providing th

If a manually-written example gets marked INCORRECT, it's often because the manual ground-truth answer was very short/terse while the model's answer is longer but still correct in substance (or vice versa) — a good reminder that automated grading is a helpful signal, not a perfect substitute for reading the actual answers yourself.

### Summary of changes made in this version

- Replaced `kaggle_secrets` and a hardcoded Kaggle PDF path with **OpenRouter** + a **Colab file-upload widget**, so this notebook works with any PDF you provide.
- Replaced deprecated import paths (`langchain.chat_models`, `langchain.document_loaders`, `langchain.vectorstores`, `langchain.embeddings`, `langchain.indexes`) with their current homes in `langchain_openai` and `langchain_community`.
- Replaced `VectorstoreIndexCreator` + `RetrievalQA` (deprecated) with the current LCEL pattern: `InMemoryVectorStore` + `create_stuff_documents_chain` + `create_retrieval_chain`.
- Replaced `QAGenerateChain` (deprecated) with a small **Pydantic `QAPair` model + `with_structured_output()`** helper for generating question/answer pairs from your document.
- Replaced `QAEvalChain` (deprecated) with a small **Pydantic `Grade` model + `with_structured_output()`** helper for grading predictions as CORRECT/INCORRECT with a reasoning string.
- Replaced the old `langchain.debug = True` global flag with the current `langchain.globals.set_debug(True)`.
- Used OpenRouter for **both chat and embeddings** through a single API key (`openai/gpt-4o-mini` and `openai/text-embedding-3-small` by default — swap either for any other OpenRouter model).

# <div style="box-shadow: rgba(240, 46, 170, 0.4) -5px 5px inset, rgba(240, 46, 170, 0.3) -10px 10px inset, rgba(240, 46, 170, 0.2) -15px 15px inset, rgba(240, 46, 170, 0.1) -20px 20px inset, rgba(240, 46, 170, 0.05) -25px 25px inset; padding:20px; font-size:30px; font-family: consolas; display:fill; border-radius:15px; color: rgba(240, 46, 170, 0.7)"> <b> ༼⁠ ⁠つ⁠ ⁠◕⁠‿⁠◕⁠ ⁠༽⁠つ Thank You!</b></div>
